# Этап B: выдачи всех поисковых конфигураций

Запускается после того, как набор запросов собран целиком: синтетические
с этапа A плюс пятьдесят написанных руками, с проверенной разметкой.

Подключить **три** входа: датасет `ru-fin-retrieval`, датасет
`ru-fin-queries` и вывод ноутбука этапа A (вкладка **Your Work**).
меню **Settings** → **Accelerator** → **GPU T4 ×2**; там же проверить,
что интернет включён (пункт читается как **Turn off internet**).


In [ ]:
!pip install -q -U sentence-transformers
!cp -r /kaggle/input/ru-fin-retrieval/* /kaggle/working/
%cd /kaggle/working


## Где лежат эмбеддинги этапа A

Путь зависит от того, как назван ноутбук этапа A, поэтому он ищется,
а не вписывается руками.


In [ ]:
import glob
found = glob.glob('/kaggle/input/*/embeddings')
print('найдено:', found)
EMB = found[0] if found else None
print('используем:', EMB)
assert EMB, 'вывод этапа A не подключён — добавьте его через + Add Input'


## Прогон

Около полутора часов; больше всего времени уходит на переранжирование
(двести запросов по пятьдесят кандидатов).


In [ ]:
!python run_phase_b.py \
    --acts /kaggle/input/ru-fin-retrieval/acts.jsonl.gz \
    --queries /kaggle/input/ru-fin-queries/queries.jsonl \
    --embeddings {EMB} \
    --out /kaggle/working/runs


## Что получилось


In [ ]:
import json, os
r = json.load(open('/kaggle/working/runs/report_phase_b.json'))
print('запросов:', r['queries'])
for x in r['runs']:
    print(f"   {x['config']:<12} {x['retrieval']}")
size = sum(os.path.getsize(os.path.join('/kaggle/working/runs', f))
           for f in os.listdir('/kaggle/working/runs'))
print(f'\nвсего {size/1048576:.1f} МБ — скачать папку runs целиком')


Скачать папку `runs` и положить в проект как `data/runs`.
Дальше на маке: `make metrics`, `make errors`, `make latency`.
